# Dependencies

In [ ]:
!python -m pip install nltk

# Helpful functions

In [1]:
import nltk
from nltk.sem.logic import LogicParser
from nltk.inference import ResolutionProver
import regex as re  

In [27]:
parser = LogicParser()
prover = ResolutionProver()

def translate_fol(fol_text: str):
    # '-' --> '_'
    fol_text = re.sub(r'(?<=[a-zA-Z0-9])-(?=[a-zA-Z0-9])', '_', fol_text)

    replacements = {
        '∀': 'all ', 
        '∃': 'exists ',
        '∧': '&', 
        '∨': '|',
        '⊕': '^',
        '¬': '-',
        '→': '->', 
        '⟷': '<->',
        '↔': '<->'
    }
    for k, v in replacements.items():
        fol_text = fol_text.replace(k, v)

    # Add dot: "all x (P(x))" --> "all x. (P(x))"
    fol_text = re.sub(r'(all|exists)\s+([a-z0-9]+)\s*', r'\1 \2. ', fol_text)

    # Fix prover9 constants name (eg: "yuri" -> "c_yuri")
    words = re.findall(r'\b[a-z][a-zA-Z0-9_]*\b', fol_text)
    reserved_words = {'all', 'exists', 'u', 'v', 'w', 'x', 'y', 'z'}
    
    for w in set(words):
        if w not in reserved_words:
            fol_text = re.sub(fr'\b{w}\b', f'c_{w}', fol_text)
    return fol_text

def is_valid_fol(fol_list):
    try:
        for line in fol_list:
            if line.strip():
                parser.parse(line)
        return True
    except Exception:
        return False

In [46]:
def check_data(data: dict):
    '''
    This function checks if a data sample is correct by checking:
        - If the syntax is correct.
        - If the label prediction (T/F/Uncertain) matches the true label

    Args:
        data: dict ({"id", "natural", "fol", "label"})
    Returns: 
        None/True/False (invalid syntax/correct/wrong prediction)
    '''
    # Read fol strings
    fol_list = data["fol"]
    translated_premises = [translate_fol(p) for p in fol_list[:-1]]
    translated_conclusion = translate_fol(fol_list[-1])

    if (not is_valid_fol(translated_premises) or not is_valid_fol([translated_conclusion])):
        return None
    
    try:
        parsed_premises = [parser.parse(p) for p in translated_premises]
        parsed_conclusion = parser.parse(translated_conclusion)
    except Exception as e:
        return None

    # Check conclusion
    predicted = "Uncertain"

    is_true = ResolutionProver().prove(parsed_conclusion, parsed_premises)
    if is_true:
        predicted = "True"

    is_false = ResolutionProver().prove(parsed_conclusion.negate(), parsed_premises)
    if is_false:
        predicted = "False"

    if (predicted == data["label"]):
        return True
    return False

def check_data_list(data_list: list):
    '''
    This function checks if a list of data samples is correct by checking:
        - If the syntax is correct.
        - If the label prediction (T/F/Uncertain) matches the true label

    Args:
        data: list ([{"id", "natural", "fol", "label"}, ...])
    Returns: 
        [...], [...]: list[int], list[int] (list of story_ids that have invalid syntax and are predicted incorrectly, respectively)
    '''
    predicted_wrong_data = [d["id"] for d in data_list if check_data(d) == False]
    invalid_data = [d["id"] for d in data_list if check_data(d) is None]
    return predicted_wrong_data, invalid_data

# Check your data

### 1. Load your file

Before testing, remember to re-run the block below everytime you make changes in your json file.

In [34]:
import json 

file_name = "Khoi.json" # <--- Change here

with open(file_name, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

data = {}
for d in raw_data:
    data[d["id"]] = d

### 2. Check 1 sample

In [ ]:
id = "folio_train_530" # <---- Change here

result = check_data(data[id])
if result == True: 
    print(f"✅ Data is Correct!")
elif result == False:
    print(f"❌ Data is predicted Wrong!!!")
elif result is None:
    print("❌ Invalid syntax!!!")

❌ Data is predicted Wrong!!!


### 3. Check the whole data

In [ ]:
wrong_data, invalid_data = check_data_list(raw_data)
print("Ids of Predicted wrong data:", wrong_data)
print("Ids of invalid syntax data:", invalid_data)

# Warning: It may take very long time or get stuck somewhere when running